# FerroAnalytics – Pruebas Fase II

Este notebook documenta y prueba los módulos nuevos de la Fase II, en el orden en que se construyeron. No modifica ni vuelve a ejecutar `notebooks/pruebas_fase1.ipynb`: esa Fase I queda cerrada y sus binarios en `data/binarios/` no se tocan aquí.

| Módulo | Responsabilidad |
|---|---|
| `excepciones.py` | Jerarquía de errores propios y logging en `data/logs/` |
| `indices.py` | Índices binarios (`.idx`) para búsqueda y upsert por `seek` |
| `clasificacion.py` | Clasificación ABC-XYZ de productos |
| `prediccion.py` | Series mensuales, comparación de modelos y punto de reorden |

Los análisis de la Fase II usan la fecha del último movimiento como fecha de referencia (no `date.today()`), y se ejecutan sobre el histórico sintético de 24 meses (`data/binarios_historico/`), generado con `scripts/generar_datos.py`: los ~260 movimientos de la Fase I no alcanzan para clasificar ni predecir demanda.

## 0. Configuración del entorno

Regeneramos el histórico sintético en un directorio de binarios propio de este notebook, para partir siempre del mismo estado (semilla fija) sin tocar `data/binarios/` (Fase I) ni el `data/binarios_historico/` que pueda estar usando el dashboard.

In [1]:
import sys, os, shutil

RAIZ = os.path.abspath('..')
os.chdir(RAIZ)
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)

DIR_BIN = os.path.join('data', 'binarios_pruebas_fase2')
if os.path.exists(DIR_BIN):
    shutil.rmtree(DIR_BIN)
os.makedirs(DIR_BIN)
os.environ['FERRO_BINARIOS'] = DIR_BIN

from scripts.generar_datos import generar, cargar_en_binarios, FECHA_INICIO, FECHA_FIN, SEMILLA, DIRECTORIO_SALIDA

resumen_generacion = generar(FECHA_INICIO, FECHA_FIN, SEMILLA, DIRECTORIO_SALIDA)
cargados = cargar_en_binarios(resumen_generacion['rutas'], DIR_BIN)

print(f'Directorio de binarios: {DIR_BIN}')
print(f'Período simulado: {FECHA_INICIO} a {FECHA_FIN} (semilla {SEMILLA})')
print(f'Cargados: {cargados}')

Directorio de binarios: data/binarios_pruebas_fase2
Período simulado: 2024-09-01 a 2026-08-31 (semilla 42)
Cargados: {'categorias': 6, 'productos': 52, 'movimientos': 12822}


In [2]:
import src.almacenamiento as alm
import src.importador as imp
import src.indices as idx
import src.clasificacion as clf
import src.prediccion as pred
from src.excepciones import (
    ArchivoCorrupto, DatosInsuficientes, ErrorImportacion,
    ErrorAlmacenamiento, FerroAnalyticsError,
)

productos = alm.leer_productos()
categorias = alm.leer_categorias()
movimientos = alm.leer_movimientos()

print(f'Productos  : {len(productos)}')
print(f'Categorías : {len(categorias)}')
print(f'Movimientos: {len(movimientos)}')

Productos  : 52
Categorías : 6
Movimientos: 12822


## 1. Módulo de Excepciones (`src/excepciones.py`)

Jerarquía: `FerroAnalyticsError` es la base de `ErrorImportacion`, `ErrorAlmacenamiento` (con `ArchivoCorrupto` como subclase) y `ErrorPrediccion` (con `DatosInsuficientes` como subclase). Cada excepción se registra automáticamente en `data/logs/ferroanalytics.log` al crearse.

In [3]:
assert issubclass(ArchivoCorrupto, ErrorAlmacenamiento)
assert issubclass(ErrorAlmacenamiento, FerroAnalyticsError)
assert issubclass(DatosInsuficientes, FerroAnalyticsError)
assert issubclass(ErrorImportacion, FerroAnalyticsError)
print('Jerarquía de excepciones verificada:')
print('  ArchivoCorrupto -> ErrorAlmacenamiento -> FerroAnalyticsError')
print('  DatosInsuficientes -> ErrorPrediccion -> FerroAnalyticsError')
print('  ErrorImportacion -> FerroAnalyticsError')

Jerarquía de excepciones verificada:
  ArchivoCorrupto -> ErrorAlmacenamiento -> FerroAnalyticsError
  DatosInsuficientes -> ErrorPrediccion -> FerroAnalyticsError
  ErrorImportacion -> FerroAnalyticsError


### 1.1 Logging automático

Provocamos un `ErrorImportacion` (un CSV sin las columnas requeridas) y confirmamos que el mensaje quedó escrito en `data/logs/ferroanalytics.log`.

In [4]:
import csv, tempfile

with tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False, newline='') as tmp:
    csv.writer(tmp).writerow(['id'])  # falta la columna 'nombre'
    ruta_csv_mala = tmp.name

try:
    imp.importar_categorias(ruta_csv_mala)
    lanzo_importacion = False
except ErrorImportacion as e:
    lanzo_importacion = True
    mensaje = str(e)

print('Se lanzó ErrorImportacion:', lanzo_importacion)
print('Mensaje:', mensaje)

with open(os.path.join('data', 'logs', 'ferroanalytics.log'), encoding='utf-8') as f:
    ultima_linea = f.readlines()[-1]
print('Última línea del log:', ultima_linea.strip())

assert lanzo_importacion and mensaje in ultima_linea
print('\nAserción OK – la excepción quedó registrada en el log.')

Se lanzó ErrorImportacion: True
Mensaje: El archivo '/var/folders/0w/3hrqfd0s20q4s45b9wp7pjr00000gn/T/tmpd_fbw75l.csv' no tiene las columnas requeridas: ['nombre']
Última línea del log: 2026-09-16 11:41:54,217 ERROR    ferroanalytics: ErrorImportacion: El archivo '/var/folders/0w/3hrqfd0s20q4s45b9wp7pjr00000gn/T/tmpd_fbw75l.csv' no tiene las columnas requeridas: ['nombre']

Aserción OK – la excepción quedó registrada en el log.


### 1.2 Detección de archivos binarios corruptos

`ArchivoCorrupto` se lanza cuando el tamaño de un `.dat` (o `.idx`) no es múltiplo del tamaño de registro. Lo probamos sobre una copia aislada en un directorio temporal, para no afectar los binarios que usa el resto del notebook.

In [5]:
import tempfile as _tempfile

dir_temporal = _tempfile.mkdtemp()
ruta_prod_temporal = os.path.join(dir_temporal, 'productos.dat')
shutil.copy(alm.RUTA_PRODUCTOS, ruta_prod_temporal)

with open(ruta_prod_temporal, 'r+b') as f:
    f.truncate(os.path.getsize(ruta_prod_temporal) - 3)  # 3 bytes menos que un registro completo

try:
    alm.leer_productos(ruta_prod_temporal)
    lanzo_corrupto = False
except ArchivoCorrupto as e:
    lanzo_corrupto = True
    print('Se lanzó ArchivoCorrupto:', e)

assert lanzo_corrupto
print('\nAserción OK – un .dat truncado se detecta en vez de leerse mal en silencio.')

shutil.rmtree(dir_temporal)

Se lanzó ArchivoCorrupto: '/var/folders/0w/3hrqfd0s20q4s45b9wp7pjr00000gn/T/tmpsv_xls54/productos.dat' tiene un tamaño (3949 bytes) que no es múltiplo del registro (76 bytes); el archivo puede estar truncado.

Aserción OK – un .dat truncado se detecta en vez de leerse mal en silencio.


## 2. Módulo de Índices Binarios (`src/indices.py`)

`productos.idx` y `movimientos.idx`/`movimientos_fecha.idx` mapean clave → posición (offset en bytes) en el `.dat` correspondiente, ordenados por clave. Permiten ubicar un registro con búsqueda binaria y actualizarlo con `seek` en vez de reescribir todo el archivo.

In [6]:
indice_productos = idx.leer_indice_productos()
codigos_ordenados = [c for c, _ in indice_productos]

assert codigos_ordenados == sorted(codigos_ordenados)
assert len(indice_productos) == len(productos)
print(f'productos.idx: {len(indice_productos)} entradas, ordenado por código ✓')

indice_mov = idx.leer_indice_movimientos()
ids_ordenados = [i for i, _ in indice_mov]
assert ids_ordenados == sorted(ids_ordenados)
assert len(indice_mov) == len(movimientos)
print(f'movimientos.idx: {len(indice_mov)} entradas, ordenado por id ✓')

productos.idx: 52 entradas, ordenado por código ✓
movimientos.idx: 12822 entradas, ordenado por id ✓


### 2.1 Búsqueda binaria

`buscar_producto`/`buscar_movimiento` hacen `seek` directamente sobre el archivo de índice (sin cargarlo completo) para ubicar la posición del registro en el `.dat`. Verificamos que la posición encontrada corresponde de verdad a ese código/id.

In [7]:
codigo_muestra = productos[0].codigo
posicion = idx.buscar_producto(codigo_muestra)

with open(alm.RUTA_PRODUCTOS, 'rb') as f:
    f.seek(posicion)
    from src.modelos import ProductoAnalitico, TAMANO_PRODUCTO
    leido = ProductoAnalitico.desde_bytes(f.read(TAMANO_PRODUCTO))

assert leido.codigo == codigo_muestra
print(f"buscar_producto('{codigo_muestra}') -> posición {posicion}, registro leído: {leido.codigo} ✓")
assert idx.buscar_producto('CODIGO-INEXISTENTE') is None
print("buscar_producto con código inexistente -> None ✓")

buscar_producto('TORN-M4') -> posición 0, registro leído: TORN-M4 ✓
buscar_producto con código inexistente -> None ✓


### 2.2 Upsert con `seek` (no reescribe todo el archivo)

Al actualizar un producto existente, `guardar_productos` ubica su posición con el índice y sobreescribe solo ese registro. La posición no debería cambiar.

In [8]:
pos_antes = idx.buscar_producto(codigo_muestra)
original = [p for p in productos if p.codigo == codigo_muestra][0]

actualizado = ProductoAnalitico(codigo_muestra, original.nombre, original.id_categoria,
                                 original.precio_unitario, 12345, original.stock_minimo, '2026-01-01')
resultado_upsert = alm.guardar_productos([actualizado])
pos_despues = idx.buscar_producto(codigo_muestra)

print('Resultado del upsert:', resultado_upsert)
print(f'Posición antes: {pos_antes}  |  después: {pos_despues}')

nuevo_stock = [p for p in alm.leer_productos() if p.codigo == codigo_muestra][0].stock_actual
assert resultado_upsert == {'insertados': 0, 'actualizados': 1}
assert pos_antes == pos_despues
assert nuevo_stock == 12345
print('\nAserción OK – el upsert actualizó en su lugar (seek), sin mover la posición.')

Resultado del upsert: {'insertados': 0, 'actualizados': 1}
Posición antes: 0  |  después: 0

Aserción OK – el upsert actualizó en su lugar (seek), sin mover la posición.


### 2.3 `reconstruir_indices()`

Borra y regenera los tres índices a partir de los `.dat` actuales; útil si un `.idx` se pierde o queda desincronizado.

In [9]:
for ruta in (idx._ruta_indice(alm.RUTA_PRODUCTOS), idx._ruta_indice(alm.RUTA_MOVIMIENTOS),
             idx._ruta_indice(alm.RUTA_MOVIMIENTOS, '_fecha')):
    os.remove(ruta)

resumen_indices = idx.reconstruir_indices()
print('Índices reconstruidos:', resumen_indices)

assert resumen_indices['productos'] == len(alm.leer_productos())
assert resumen_indices['movimientos'] == len(alm.leer_movimientos())
print('\nAserción OK – los índices reconstruidos coinciden con los .dat.')

Índices reconstruidos: {'productos': 52, 'movimientos': 12822, 'movimientos_fecha': 12822}

Aserción OK – los índices reconstruidos coinciden con los .dat.


## 3. Módulo de Clasificación ABC-XYZ (`src/clasificacion.py`)

ABC por valor de consumo (unidades vendidas × precio actual), con cortes acumulados en 80% y 95%. XYZ por el coeficiente de variación de la demanda mensual: `X` si CV < 0.5, `Y` si 0.5 ≤ CV ≤ 1, `Z` si CV > 1.

In [10]:
from datetime import datetime as _dt

ref = clf.fecha_referencia(movimientos)
esperado_ref = max(_dt.strptime(m.fecha, '%Y-%m-%d').date() for m in movimientos)
print('Fecha de referencia (último movimiento):', ref)
assert ref == esperado_ref

clasificacion = clf.clasificar_abc_xyz(productos, movimientos)
resumen_matriz = clf.resumen_matriz(clasificacion)

print(f'\n{len(clasificacion)} productos clasificados en {len(resumen_matriz)} celdas de la matriz ABC-XYZ:\n')
print(f"  {'CELDA':<6}{'PRODUCTOS':>11}{'VALOR CONSUMO (L)':>20}")
for c in resumen_matriz:
    print(f"  {c['celda']:<6}{c['num_productos']:>11}{c['valor_consumo_total']:>20,.2f}")

abc_solo = clf.clasificar_abc(productos, movimientos)
print(f"Porcentaje acumulado del último producto (debe ser ~100%): {abc_solo[-1]['porcentaje_acumulado']}%")

assert all(r['clase_abc'] in {'A', 'B', 'C'} for r in clasificacion)
assert all(r['clase_xyz'] in {'X', 'Y', 'Z'} for r in clasificacion)
assert abs(abc_solo[-1]['porcentaje_acumulado'] - 100.0) < 0.05
assert sum(c['num_productos'] for c in resumen_matriz) == len(productos)
print('\nAserción OK – toda la clasificación es consistente (clases válidas, 100% acumulado, conteos completos).')

Fecha de referencia (último movimiento): 2026-08-31

52 productos clasificados en 9 celdas de la matriz ABC-XYZ:

  CELDA   PRODUCTOS   VALOR CONSUMO (L)
  AX              9          147,317.60
  AY              4           18,782.60
  AZ              1            5,390.00
  BX             11           22,243.57
  BY              3            5,682.40
  BZ              4            5,945.00
  CX             12            6,017.68
  CY              6            3,548.60
  CZ              2            1,391.00
Porcentaje acumulado del último producto (debe ser ~100%): 100.0%

Aserción OK – toda la clasificación es consistente (clases válidas, 100% acumulado, conteos completos).


## 4. Módulo de Predicción de Demanda (`src/prediccion.py`)

Compara cuatro modelos de pronóstico mediante *backtesting* (MAE y MAPE dejando los últimos meses como prueba): ingenuo, media móvil, suavizado exponencial simple y una regresión con tendencia + estacionalidad mensual (scikit-learn). `productos_prioritarios()` selecciona los productos A/X (alto valor y demanda estable): los mejores candidatos para pronosticar por producto en vez de solo por categoría.

In [11]:
ax = pred.productos_prioritarios(productos, movimientos)
print(f'Productos A/X ({len(ax)}):', ax)

codigo_ax = ax[0]
resultado_prod = pred.pronosticar_producto(productos, movimientos, codigo_ax, n=3)

print(f"\nComparación de modelos para '{codigo_ax}' (backtest, MAE ascendente):")
for m in resultado_prod['comparacion_modelos']:
    print(f"  {m['modelo']:<22} MAE={m['mae']:>9.3f}  MAPE={m['mape']}")

maes = [m['mae'] for m in resultado_prod['comparacion_modelos']]
assert maes == sorted(maes)
assert len(resultado_prod['comparacion_modelos']) == 4
print(f"\nMejor modelo: {resultado_prod['mejor_modelo']}")
print(f"Pronóstico próximos meses: {list(zip(resultado_prod['meses_pronosticados'], resultado_prod['pronostico']))}")
print('\nAserción OK – los 4 modelos se evaluaron y quedaron ordenados por MAE.')

Productos A/X (9): ['PINT-BL', 'LIJAD-1', 'CABL-12', 'TOMA-SNC', 'MART-2A', 'PINT-BG', 'LLAV-12', 'PINT-GR', 'NIVE-60']

Comparación de modelos para 'PINT-BL' (backtest, MAE ascendente):
  regresion              MAE=   25.704  MAPE=29.43
  suavizado_exponencial  MAE=   30.588  MAPE=34.67
  media_movil            MAE=   39.000  MAPE=43.58
  ingenuo                MAE=   40.333  MAPE=44.99

Mejor modelo: regresion
Pronóstico próximos meses: [((2026, 9), 112.0), ((2026, 10), 122.5), ((2026, 11), 139.0)]

Aserción OK – los 4 modelos se evaluaron y quedaron ordenados por MAE.


### 4.1 Punto de reorden y stock de seguridad

A partir de la demanda diaria observada del producto (incluye los días sin ventas), con un tiempo de entrega y un nivel de servicio dados.

In [12]:
print(f"Demanda diaria media       : {resultado_prod['demanda_diaria_media']} uds")
print(f"Desviación diaria           : {resultado_prod['demanda_diaria_desviacion']} uds")
print(f"z (nivel de servicio 95%)   : {resultado_prod['z']}")
print(f"Stock de seguridad          : {resultado_prod['stock_seguridad']} uds")
print(f"Punto de reorden            : {resultado_prod['punto_reorden']} uds")

assert resultado_prod['punto_reorden'] >= resultado_prod['stock_seguridad'] >= 0
print('\nAserción OK – punto de reorden >= stock de seguridad >= 0.')

Demanda diaria media       : 3.46 uds
Desviación diaria           : 3.485 uds
z (nivel de servicio 95%)   : 1.645
Stock de seguridad          : 15.17 uds
Punto de reorden            : 39.38 uds

Aserción OK – punto de reorden >= stock de seguridad >= 0.


### 4.2 Pronóstico por categoría

In [13]:
id_categoria_muestra = categorias[0].id
resultado_cat = pred.pronosticar_categoria(productos, movimientos, categorias, id_categoria_muestra, n=3)

print(f"Categoría: {resultado_cat['nombre_categoria']}")
print(f"Mejor modelo: {resultado_cat['mejor_modelo']}")
print(f"Pronóstico próximos meses: {list(zip(resultado_cat['meses_pronosticados'], resultado_cat['pronostico']))}")

assert len(resultado_cat['pronostico']) == 3
print('\nAserción OK – el pronóstico por categoría trae 3 meses futuros.')

Categoría: Tornillería
Mejor modelo: media_movil
Pronóstico próximos meses: [((2026, 9), 4213.67), ((2026, 10), 4213.67), ((2026, 11), 4213.67)]

Aserción OK – el pronóstico por categoría trae 3 meses futuros.


### 4.3 Datos insuficientes

Con la muestra pequeña de la Fase I (~260 movimientos en total, 2 a 5 por producto) no alcanza para pronosticar: `pronosticar_producto` debe rechazarla con `DatosInsuficientes` en vez de dar un resultado poco confiable.

In [14]:
productos_f1 = alm.leer_productos(os.path.join('data', 'binarios', 'productos.dat'))
movimientos_f1 = alm.leer_movimientos(os.path.join('data', 'binarios', 'movimientos.dat'))

lanzo_datos_insuficientes = False
if productos_f1 and movimientos_f1:
    try:
        pred.pronosticar_producto(productos_f1, movimientos_f1, productos_f1[0].codigo)
    except DatosInsuficientes as e:
        lanzo_datos_insuficientes = True
        print('Se lanzó DatosInsuficientes (esperado):', e)
else:
    print('No hay binarios de la Fase I cargados en data/binarios/; se omite esta prueba.')
    lanzo_datos_insuficientes = True  # nada que objetar si la Fase I no está cargada en este entorno

assert lanzo_datos_insuficientes
print('\nAserción OK – la muestra de la Fase I no alcanza y se rechaza con un mensaje claro.')

Se lanzó DatosInsuficientes (esperado): Producto 'TORN-M4': solo 4 de 18 meses tienen ventas registradas; no alcanza para un pronóstico confiable.

Aserción OK – la muestra de la Fase I no alcanza y se rechaza con un mensaje claro.


## 5. Resumen de Pruebas

In [15]:
print('=' * 55)
print('  RESUMEN – FerroAnalytics Fase II')
print('=' * 55)

pruebas = [
    ('Jerarquía de excepciones correcta',              True),
    ('ErrorImportacion se registra en el log',         mensaje in ultima_linea),
    ('ArchivoCorrupto detecta binario truncado',        lanzo_corrupto),
    ('productos.idx ordenado y completo',               codigos_ordenados == sorted(codigos_ordenados) and len(indice_productos) == len(productos)),
    ('movimientos.idx ordenado y completo',             ids_ordenados == sorted(ids_ordenados) and len(indice_mov) == len(movimientos)),
    ('Upsert con seek no mueve la posición',            pos_antes == pos_despues and nuevo_stock == 12345),
    ('reconstruir_indices coincide con los .dat',       resumen_indices['productos'] == len(alm.leer_productos())),
    ('Clasificación ABC-XYZ consistente (100% acum.)',  abs(abc_solo[-1]['porcentaje_acumulado'] - 100.0) < 0.05),
    ('Los 4 modelos de pronóstico se comparan por MAE', maes == sorted(maes) and len(resultado_prod['comparacion_modelos']) == 4),
    ('Punto de reorden >= stock de seguridad >= 0',     resultado_prod['punto_reorden'] >= resultado_prod['stock_seguridad'] >= 0),
    ('Pronóstico por categoría trae n meses',           len(resultado_cat['pronostico']) == 3),
    ('Datos insuficientes (Fase I) se rechazan',        lanzo_datos_insuficientes),
]

for nombre, resultado in pruebas:
    estado = 'PASS' if resultado else 'FAIL'
    print(f'  [{estado}]  {nombre}')

n_pass = sum(1 for _, r in pruebas if r)
print()
print(f'  Resultado: {n_pass}/{len(pruebas)} pruebas pasadas')
print('=' * 55)

shutil.rmtree(DIR_BIN)
print(f"\nLimpieza: se eliminó el directorio de prueba '{DIR_BIN}'.")

  RESUMEN – FerroAnalytics Fase II
  [PASS]  Jerarquía de excepciones correcta
  [PASS]  ErrorImportacion se registra en el log
  [PASS]  ArchivoCorrupto detecta binario truncado
  [PASS]  productos.idx ordenado y completo
  [PASS]  movimientos.idx ordenado y completo
  [PASS]  Upsert con seek no mueve la posición
  [PASS]  reconstruir_indices coincide con los .dat
  [PASS]  Clasificación ABC-XYZ consistente (100% acum.)
  [PASS]  Los 4 modelos de pronóstico se comparan por MAE
  [PASS]  Punto de reorden >= stock de seguridad >= 0
  [PASS]  Pronóstico por categoría trae n meses
  [PASS]  Datos insuficientes (Fase I) se rechazan

  Resultado: 12/12 pruebas pasadas

Limpieza: se eliminó el directorio de prueba 'data/binarios_pruebas_fase2'.
